# Cramér–Rao bounds

In [ ]:
import jax.numpy as jnp
import numpy as np
from master.fft import *
import jax
from master.symmetry import *
from master.phase_retrieval import *
from master.obj import *
from master.noise import *
from master.mle import *
import master.device
from jax import lax
from master.pies import *
from master.ptychography import *
from master.imaging import make_circular_mask
from master.spotgen import *
from master.estimator_bounds import *

jax.config.update("jax_enable_x64", False)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = master.device.setup_device(device_id=1)

In [ ]:
from master.loader import Loader

base_path = "/data/multispot_pty/pty-64-jitter-all-phs/"
loader = Loader(base_path)

fig_path = "../docs/thesis/figures"
SAVE_FIGS = True

In [ ]:
from master.digest import make_array_digest
from master.probes import construct_probe_fields, construct_probe_inf_probe_approx
from pty_defs import ProbeParams, PtySim

pixel_intensity = 1e6
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
est_id = "ORACLE-second-order-ml"
est_label = "ORACLE+ML"


x_true, d, probes, spectral_support, _, measurement_params, measurement_hash_meas = (
    loader.get_measurement_with_probes(meas_id)
)

P_spectral_support = lambda x: ifft2_shifted(fft2_shifted(x) * spectral_support)
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
    loader.get_estimate_with_x_est_raw(meas_id, est_id)
)

# Check that measurement corresponds to reconstruction
assert measurement_hash_est == measurement_hash_meas
sim_params = simulation_params["simulation"]
probe_dict = simulation_params["probe"]
solver_params = simulation_params["solver"]
ml_params = solver_params["ml_stage"]

N_crop = sim_params["N_crop"]
N_pad = sim_params["N_pad"]
N_photons = sim_params["N_photons"]
N_samples = sim_params["N_samples"]
N_center_crop = N_crop
M = sim_params["M"]

print(N_crop, N_pad)
NA_det = sim_params["NA_det"]

N_iter_ML = ml_params["N_iter_ML"]

probe_params = ProbeParams.from_dict(probe_dict)
print(probe_params)
pty_sim = PtySim(N_crop, N_pad, NA_det)
x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = pty_sim.create_axes()
A, A_dagger, F_pupil, pupil_support = pty_sim.get_ops_and_pupil()

plot_intensity_phase(F_pupil)

obj_support = pty_sim.obj_support

N_d_crop = d.shape[-1]

r_extent_cropped = list(
    create_physical_axes(center_crop(x_true, N_crop), pixel_size=1)
)[-2]
print(k_extent)
print(
    "Unit intensity probes and object digest: ",
    make_array_digest(probes),
    make_array_digest(x_true),
)

x_true_est = P_spectral_support(x_true)
plot_intensity_phase(probes[0])

x_true.shape, d.shape, x_est.shape, simulation_params["simulation"], N_d_crop
plot_intensity_phase(fft2_shifted(x_true_est), use_log=True)
print(F_pupil.shape, pupil_support.shape)

In [ ]:
prod_ph = probes * x_true[None, :, :]
ys = A(prod_ph)
plot_intensity_phase(prod_ph[0], r_extent, label="y")
plot_intensity_phase(ys[0], r_extent, label="y", use_log=True)

In [ ]:
def print_shapes(prod_ph, y_ph, probes):
    print("prod_ph.shape:", prod_ph.shape)
    print("y_ph.shape:   ", y_ph.shape)
    print("probes.shape: ", probes.shape)

    print("prod_ph.ndim:", prod_ph.ndim)
    print("y_ph.ndim:   ", y_ph.ndim)
    print("probes.ndim: ", probes.ndim)

    g = jnp.exp(-2j * jnp.angle(y_ph))
    H = ifft2_shifted(g)

    print("g.shape:     ", g.shape)
    print("H.shape:     ", H.shape)


print_shapes(prod_ph, ys, probes)

## 1. Bouchet style bounds on CRB
First some fundamental bounds are considered as described in Fundamental Bounds on the Precision of Classical Phase Microscopes. Assume an object field with 
$$x_k = \sqrt{n_k}e^{i\phi_k}$$
so that $n_k$ photons (on average) are indicent on the associated area of pixel $k$. It can be derived from the diagonal FIM terms that
$$\operatorname{Var} (\hat{n}_k) \geq n_k$$
$$\operatorname{Var} (\hat{\phi}_k) \geq \frac{1}{4n_k}$$

## 1.1. Lower bound on CRB for $T_k$

In [ ]:
# n_k = jnp.sum(jnp.abs(prod_ph) ** 2, -3)
# I_k = jnp.sum(jnp.abs(probes) ** 2, axis=-3)  # incident counts
# n_k = jnp.sum(jnp.abs(prod_ph) ** 2, axis=-3)  # transmitted counts
# eps = 1e-12

# # How much intensity is transmitted
# T_k = n_k / (I_k + eps)

# # Just poisson statistics
# T_k_var_bouchet = T_k / (I_k + eps)
# T_k_std_bouchet = jnp.sqrt(T_k / (I_k + eps))
# plot(crop(T_k_std_bouchet), label=r"$[\sigma_{\hat{T}_k}]^{\text{Bouchet}}$")

In [ ]:
crop = lambda x: center_crop(x * obj_support, N_crop + 2 * N_pad)
prod_ph = probes * x_true_est[None, :, :]
ys = A(prod_ph)

# A dirac delta at some position gives a phase ramp in Fourier spectrum or a flat intensity field
delta = jnp.zeros(prod_ph.shape[-2:], prod_ph.dtype).at[0, 0].set(1)
eff = jnp.sum(jnp.abs(A(delta)) ** 2)  # (A†A)_kk
# eff = jnp.sum(jnp.abs(A_dagger(A(delta))) ** 2)          # (A†A)_kk
print(eff)
n_k = eff * jnp.sum(jnp.abs(prod_ph) ** 2, -3)
phi_var_bouchet = 1 / (4 * n_k)
phi_std_bouchet = jnp.sqrt(phi_var_bouchet)


n_k_std_bouchet = jnp.abs(x_true_est) ** 2 / jnp.sqrt(
    n_k
)  # obj in object frame, same grid as n_k
plot(
    crop(n_k_std_bouchet),
    label=r"$[\sigma_{\hat{a}_k}]^{Bouchet}$",
)
plot(crop(phi_std_bouchet), label=r"$[\sigma_{\hat{\phi}_k}]^{\text{Bouchet}}$")

In [ ]:
# Consensus form
prod_ph = probes * x_true_est[None, :, :]
ys = A(prod_ph)
f_consensus = jit(
    lambda z: jnp.sum(jnp.conj(probes) * z, axis=-3)
    / jnp.sum(jnp.abs(probes) ** 2, axis=-3)
)
probes.shape
Jplus_nn_unnorm = np.zeros_like(x_true, dtype=np.complex128)
for m in range(M):
    Jplus_nn_z, Jplus_phiphi_z, Jplus_phin_z = fisher_pinv_2x2_block(
        prod_ph[m, :], ys[m, :], N_y=ys.shape[-1]
    )
    Jplus_nn_unnorm += jnp.conj(probes[m]) * Jplus_nn_z
    if m == 0:
        plot_intensity_phase(Jplus_nn_z)
Jplus_nn_lsq = np.abs(Jplus_nn_unnorm / jnp.sum(jnp.abs(probes) ** 2, axis=-3))
sigma_nn_lsq = eff * np.sqrt(Jplus_nn_lsq)
plot(sigma_nn_lsq)

In [ ]:
def pinv_2x2_cart(S, Q):
    return tuple(c / 16 for c in fisher_pty_2x2_cart(S, Q))


def pinv_2x2_polar(S, Q, x):
    n = jnp.abs(x) ** 2
    Jnn, Jpp, Jnp_ = fisher_pty_2x2_polar(S, Q, x)
    return n**2 * Jnn, Jpp / (16 * n**2), Jnp_ / 4


def ls_fs_cov(probes, ys, pad, x, pupil=1.0):
    S, Q = ptycho_SQ(probes, ys, pad, pupil)
    D2 = jnp.sum(jnp.abs(probes) ** 2, 0) ** 2
    cart = tuple(c / D2 for c in pinv_2x2_cart(S, Q))
    polar = tuple(c / D2 for c in pinv_2x2_polar(S, Q, x))
    return cart, polar


def bouchet_bounds(x, S=1.0):  # J_k ⪯ 4S I
    n_k = S * jnp.abs(x) ** 2
    return jnp.abs(x) ** 2 / jnp.sqrt(n_k), 1 / (2 * jnp.sqrt(n_k)), 0.5 / jnp.sqrt(S)

In [ ]:
from master.operators import make_crop_pad
from pty_defs import pupil_crop

N_pupil_crop = pupil_crop(F_pupil)
crop, pad = make_crop_pad(N_full=N_crop + 2 * N_pad, N_crop=N_pupil_crop)
cropped_pupil = crop(F_pupil)
S, Q = ptycho_SQ(probes, ys, pad, cropped_pupil)  # CDI: S, Q = 1.0, T / d
Jnn, Jpp, Jnp_ = fisher_pty_2x2_polar(S, Q, x_true_est)
n_var, phi_var = diag_of_2x2_inverse(Jnn, Jnp_, Jnp_, Jpp)


print(probes.shape, ys.shape, cropped_pupil.shape)
plot(obj_support * np.sqrt(n_var))
plot(obj_support * np.sqrt(phi_var))

In [ ]:
(C_uu, C_vv, C_uv), (C_nn, C_φφ, C_nφ) = ls_fs_cov(
    probes, ys, pad, x_true_est, pupil=cropped_pupil
)

plot(obj_support * np.sqrt(C_nn))
plot(obj_support * np.sqrt(C_φφ))
plot(obj_support * np.sqrt(n_var) / np.sqrt(C_nn))
plot(obj_support * np.sqrt(phi_var) / np.sqrt(C_φφ))
print(np.max(np.sqrt(n_var) / np.sqrt(C_nn)))

In [ ]:
2 / eff

## 1.2. Lower bound on CRB for phase $\phi_k$

In [ ]:
n_k = jnp.sum(jnp.abs(A_dagger(A(prod_ph))) ** 2, -3)
n_k_old = jnp.sum(jnp.abs(prod_ph) ** 2, -3)
plot(n_k_old - n_k)
phi_var_bouchet = 1 / (4 * n_k)
phi_std_bouchet = jnp.sqrt(phi_var_bouchet)
plot(
    crop(phi_std_bouchet),
    label=r"$[\sigma_{\hat{\phi}_k}]^{Bouchet}$",
)

# 2. Bounds from exact diagonal elements


In [ ]:
import jax
import jax.numpy as jnp


@jax.jit
def fisher_diag_bouchet_fast(x_ph, y_ph, probes, eps=1e-12):
    N1, N2 = x_ph.shape[-2:]
    d = N1 * N2

    n = jnp.abs(x_ph) ** 2
    phi = jnp.angle(x_ph)
    psi = jnp.angle(y_ph)

    pupil_weight = jnp.abs(F_pupil) ** 2
    g = pupil_weight[None, :, :] * jnp.exp(-2j * psi)
    H = fft2_shifted(g)

    J1 = jnp.arange(N1) - N1 // 2
    J2 = jnp.arange(N2) - N2 // 2
    J1, J2 = jnp.meshgrid(J1, J2, indexing="ij")

    I1 = jnp.mod(2 * J1, N1)
    I2 = jnp.mod(2 * J2, N2)

    I1 = (I1 + N1 // 2) % N1
    I2 = (I2 + N2 // 2) % N2

    # Correct: preserve exposure axis
    H2 = H[:, I1, I2]  # shape (M, N1, N2)

    S = jnp.exp(2j * phi) * (jnp.sqrt(d) * H2)

    ReS = jnp.real(S)
    ImS = jnp.imag(S)

    d_pupil = jnp.sum(pupil_weight)

    cos2_sum = 0.5 * (d_pupil + ReS)
    sin2_sum = 0.5 * (d_pupil - ReS)
    cos_sin_sum = 0.5 * ImS

    p = jnp.abs(probes) ** 2

    I_k = jnp.sum(p, axis=0)
    n_k = jnp.sum(n, axis=0)
    T_k = n_k / (I_k + eps)

    JTT_diag = (1.0 / ((T_k + eps) * d)) * jnp.sum(
        p * cos2_sum,
        axis=0,
    )

    Jphi_diag = jnp.sum(
        4.0 * n * sin2_sum / d,
        axis=0,
    )

    JTphi_diag = -2.0 * jnp.sum(
        p * cos_sin_sum / d,
        axis=0,
    )

    T_var_diag = 1.0 / (JTT_diag + eps)
    phi_var_diag = 1.0 / (Jphi_diag + eps)

    return JTT_diag, Jphi_diag, JTphi_diag, T_var_diag, phi_var_diag


Jn_diag, Jphi_diag, Jnphi_off, n_var_diag, phi_var_diag = fisher_diag_bouchet_fast(
    prod_ph, ys, probes
)

In [ ]:
det_j = Jn_diag * Jphi_diag - Jnphi_off**2

In [ ]:
np.mean(Jn_diag * Jphi_diag), np.mean(Jnphi_off**2)

In [ ]:
n_k_std_num_diag = jnp.sqrt(n_var_diag) * obj_support * probed
phi_k_std_num_diag = jnp.sqrt(phi_var_diag) * obj_support * probed
plot(crop(n_k_std_num_diag), label=r"$[\sigma_{\hat{n}_k}]^{Diagonal CRB}$")
plot(
    crop(phi_k_std_num_diag),
    label=r"$[\sigma_{\hat{\phi}_k}]^{Diagonal CRB}$",
)
plot(
    crop(det_j),
    label=r"$[det(J_{jj})]$",
)
# Verify that the difference between this and the optimistic bound is about sqrt(2) with some minor local variation signal
# depending on the object imaged
plot(
    crop(jnp.sqrt(n_var_diag) / T_k_std_bouchet),
    label=r"$[\sigma_{\hat{n}_k}]^{Diagonal CRB}/[\sigma_{\hat{n}_k}]^{Bouchet}$",
)
plot(
    crop(jnp.sqrt(phi_var_diag) / phi_std_bouchet),
    label=r"$[\sigma_{\hat{\phi}_k}]^{Diagonal CRB}/[\sigma_{\hat{\phi}_k}]^{Bouchet}$",
)

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax


@jax.jit
def fisher_full_ptycho_bouchet_fft_lowmem(
    prod_ph,
    y_ph,
    probes,
    eps=1e-12,
):
    """Here be dragons: this NA-limited generalization is AI-generated, its comments were kept for future reference."""
    M, N1, N2 = prod_ph.shape
    d = N1 * N2
    P = d

    # Detector intensity weight.
    # For a binary pupil this equals F_pupil.
    pupil_weight = jnp.abs(F_pupil) ** 2

    p = jnp.abs(probes) ** 2
    n = jnp.abs(prod_ph) ** 2
    theta = jnp.angle(prod_ph)

    # For a real nonnegative/binary pupil, angle(y_ph) is fine.
    # If F_pupil has phase, use the phase of the pre-pupil propagated field instead.
    psi = jnp.angle(y_ph)

    # Plus kernel: contains exp(-2j psi), gives theta_i + theta_j term.
    g_plus = pupil_weight[None, :, :] * jnp.exp(-2j * psi)
    H_plus = ifft2_shifted(g_plus)

    # Minus/difference kernel: pupil autocorrelation-like term,
    # gives theta_i - theta_j contribution.
    H_minus = ifft2_shifted(pupil_weight)

    I_inc = jnp.sum(p, axis=0)

    # Physical transfer estimate. Do not NA-limit this again;
    # NA loss is already in pupil_weight/H_plus/H_minus.
    T = jnp.sum(n, axis=0) / (I_inc + eps)

    T_flat = T.reshape(P)
    sqrt_T = jnp.sqrt(T_flat + eps)

    # Centered spatial indices.
    J1 = jnp.arange(N1) - N1 // 2
    J2 = jnp.arange(N2) - N2 // 2
    J1g, J2g = jnp.meshgrid(J1, J2, indexing="ij")

    j1 = J1g.reshape(-1)
    j2 = J2g.reshape(-1)

    # Sum indices: r_i + r_j
    S1 = j1[:, None] + j1[None, :]
    S2 = j2[:, None] + j2[None, :]

    I1_sum = (S1 + N1 // 2) % N1
    I2_sum = (S2 + N2 // 2) % N2

    # Difference indices: r_i - r_j
    D1 = j1[:, None] - j1[None, :]
    D2 = j2[:, None] - j2[None, :]

    I1_diff = (D1 + N1 // 2) % N1
    I2_diff = (D2 + N2 // 2) % N2

    # Shared across exposures.
    # Keep sqrt(d): this comes from unitary FFT normalization.
    Qminus = jnp.sqrt(d) * H_minus[I1_diff, I2_diff]

    def body(carry, m):
        JTT, Jphiphi, JTphi = carry

        Hm_plus = H_plus[m]

        # Plus kernel for this exposure.
        # Keep sqrt(d), not sqrt(sum pupil_weight).
        Qplus = jnp.sqrt(d) * Hm_plus[I1_sum, I2_sum]

        theta_m = theta[m].reshape(P)
        p_m = p[m].reshape(P)
        n_m = n[m].reshape(P)

        phase_plus = jnp.exp(1j * (theta_m[:, None] + theta_m[None, :]))
        phase_minus = jnp.exp(1j * (theta_m[:, None] - theta_m[None, :]))

        Cplus = phase_plus * Qplus
        Cminus = phase_minus * Qminus

        ReCp = jnp.real(Cplus)
        ImCp = jnp.imag(Cplus)

        ReCm = jnp.real(Cminus)
        ImCm = jnp.imag(Cminus)

        sqrt_p = jnp.sqrt(p_m + eps)
        sqrt_n = jnp.sqrt(n_m + eps)

        TT_prefactor = (
            sqrt_p[:, None] * sqrt_p[None, :] / (sqrt_T[:, None] * sqrt_T[None, :])
        )

        JTT = JTT + TT_prefactor * (ReCm + ReCp) / (2.0 * d)

        n_prefactor = sqrt_n[:, None] * sqrt_n[None, :]

        Jphiphi = Jphiphi + (2.0 / d) * n_prefactor * (ReCm - ReCp)

        Tphi_prefactor = (
            sqrt_p[:, None] * sqrt_p[None, :] * (sqrt_T[None, :] / sqrt_T[:, None])
        )

        # Full-aperture case has ImCm = 0, recovering the old expression.
        JTphi = JTphi - Tphi_prefactor * (ImCp - ImCm) / d

        return (JTT, Jphiphi, JTphi), None

    JTT0 = jnp.zeros((P, P), dtype=jnp.float32)
    Jphiphi0 = jnp.zeros((P, P), dtype=jnp.float32)
    JTphi0 = jnp.zeros((P, P), dtype=jnp.float32)

    (JTT, Jphiphi, JTphi), _ = lax.scan(
        body,
        (JTT0, Jphiphi0, JTphi0),
        jnp.arange(M),
    )

    # Optional but usually sensible: enforce numerical symmetry of same-variable blocks.
    JTT = 0.5 * (JTT + JTT.T)
    Jphiphi = 0.5 * (Jphiphi + Jphiphi.T)

    return JTT, Jphiphi, JTphi


JTT, Jphiphi, JTphi = fisher_full_ptycho_bouchet_fft_lowmem(prod_ph, ys, probes)


top = jnp.concatenate([JTT, JTphi], axis=1)
bot = jnp.concatenate([JTphi.T, Jphiphi], axis=1)
J = jnp.concatenate([top, bot], axis=0)
J.shape

In [ ]:
support_flat = obj_support.reshape(-1).astype(bool)
idx = jnp.where(support_flat)[0]
P = obj_support.size
idx_n = idx
idx_phi = idx + P
idx_full = jnp.concatenate([idx_n, idx_phi])
J_sup = J[idx_full[:, None], idx_full[None, :]]
J_sup.shape

In [ ]:
evals, evecs = jnp.linalg.eigh(J_sup)

In [ ]:
rtol = 5e-7
tol = rtol * jnp.max(evals)
evals_pinv = jnp.where(evals > tol, 1.0 / evals, 0.0)
N_support_params = jnp.sum(obj_support * 2)
print(jnp.sum(evals > tol), N_support_params)
J_nn_pinv = (evecs * evals_pinv[None, :]) @ evecs.T

In [ ]:
def extract_crb_diagonals(J_pinv, shape):
    N1, N2 = shape
    P = N1 * N2

    diag = jnp.diag(J_pinv)

    n_crb = diag[:P].reshape(N1, N2)
    phi_crb = diag[P:].reshape(N1, N2)

    return n_crb, phi_crb


n_num_crb_single_matrix, phi_num_crb_single_matrix = extract_crb_diagonals(
    J_nn_pinv, (N_crop, N_crop)
)
n_num_crb_single_matrix = pad_image(n_num_crb_single_matrix, N_pad)
phi_num_crb_single_matrix = pad_image(phi_num_crb_single_matrix, N_pad)

In [ ]:
plot(
    crop(jnp.sqrt(n_num_crb_single_matrix)),
    label=r"$[\sigma_{\hat{n}_k}]^{Exact CRB}$",
)
plot(
    crop(jnp.sqrt(phi_num_crb_single_matrix)),
    label=r"$[\sigma_{\hat{\phi}_k}]^{Exact CRB}$",
)